In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [2]:
# return_data.csv
filename = 'return_data.csv'

# find in git
if os.path.exists(filename):
    df = pd.read_csv(filename)
    print(f"Success: Loaded {filename} from local folder.")

# load google drive file version if not found
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = f'/content/drive/MyDrive/leg_one_indicators/{filename}'
        df = pd.read_csv(drive_path)
        print(f"Success: Loaded {filename} from Google Drive.")
    except Exception as e:
        print(f"Error: Could not find {filename}. Make sure it is in the same folder as this notebook.")

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Success: Loaded return_data.csv from Google Drive.


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,ticker
0,2010-01-04 00:00:00-05:00,6.487649,6.520174,6.455732,6.505279,493729600,0.0,0.0,AAPL
1,2010-01-05 00:00:00-05:00,6.523214,6.553307,6.482178,6.516527,601904800,0.0,0.0,AAPL
2,2010-01-06 00:00:00-05:00,6.516527,6.542364,6.406185,6.412873,552160000,0.0,0.0,AAPL
3,2010-01-07 00:00:00-05:00,6.436583,6.444183,6.354511,6.401018,477131200,0.0,0.0,AAPL
4,2010-01-08 00:00:00-05:00,6.392506,6.444181,6.354814,6.443573,447610800,0.0,0.0,AAPL


In [3]:
def calculate_simple_features(df):
    """
    Computes Simple Returns (Percentage Change) and other Leg A indicators.
    """
    df = df.copy()
    df = df.sort_values(['ticker', 'Date'])
    g = df.groupby('ticker')

    # 1. Returns
    # r1
    df['r1'] = g['Close'].transform(lambda x: x.pct_change(periods=1))

    # r5
    df['r5'] = g['Close'].transform(lambda x: x.pct_change(periods=5))

    # r10
    df['r10'] = g['Close'].transform(lambda x: x.pct_change(periods=10))

    # 2. Trend (EMA Difference)
    ema10 = g['Close'].transform(lambda x: x.ewm(span=10, adjust=False).mean())
    ema30 = g['Close'].transform(lambda x: x.ewm(span=30, adjust=False).mean())
    df['trend_ema_diff'] = ema10 - ema30

    # 3. Volatility
    df['vol_realized_20d'] = g['r1'].transform(lambda x: x.rolling(window=20).std())

    hl_ratio_sq = (np.log(df['High'] / df['Low'])) ** 2
    rolling_sum = hl_ratio_sq.groupby(df['ticker']).rolling(20).sum().reset_index(level=0, drop=True)
    constant = 1.0 / (4.0 * 20 * np.log(2))
    df['vol_parkinson_20d'] = np.sqrt(constant * rolling_sum)

    # 4. Liquidity (Turnover)
    df['turnover'] = df['Close'] * df['Volume']

    return df.dropna()

In [4]:
df_final = calculate_simple_features(df)
df_final.head(100)

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,ticker,r1,r5,r10,trend_ema_diff,vol_realized_20d,vol_parkinson_20d,turnover
20,2010-02-02 00:00:00-05:00,5.955093,5.967555,5.878188,5.953572,698342400,0.0,0.0,AAPL,0.005803,-0.048946,-0.089193,-0.183452,0.022815,0.019538,4.157632e+09
21,2010-02-03 00:00:00-05:00,5.932600,6.085497,5.909802,6.056012,615328000,0.0,0.0,AAPL,0.017206,-0.041611,-0.059037,-0.176682,0.023283,0.019876,3.726434e+09
22,2010-02-04 00:00:00-05:00,5.980018,6.029870,5.823170,5.837761,757652000,0.0,0.0,AAPL,-0.036039,-0.036329,-0.076993,-0.195029,0.024267,0.020224,4.422991e+09
23,2010-02-05 00:00:00-05:00,5.855390,5.957828,5.801282,5.941413,850306800,0.0,0.0,AAPL,0.017755,0.017703,-0.011581,-0.194625,0.024767,0.020451,5.052024e+09
24,2010-02-08 00:00:00-05:00,5.948405,6.014975,5.897034,5.900681,478270800,0.0,0.0,AAPL,-0.006856,-0.003133,-0.044073,-0.196811,0.024661,0.020538,2.822124e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2010-06-18 00:00:00-04:00,8.275610,8.359203,8.250380,8.330933,784621600,0.0,0.0,AAPL,0.008092,0.081101,0.070754,0.187967,0.018077,0.017274,6.536630e+09
116,2010-06-21 00:00:00-04:00,8.440965,8.481089,8.168607,8.212379,776490400,0.0,0.0,AAPL,-0.014230,0.062490,0.076632,0.205729,0.018460,0.016392,6.376834e+09
117,2010-06-22 00:00:00-04:00,8.272872,8.388684,8.252810,8.324243,717262000,0.0,0.0,AAPL,0.013621,0.054527,0.098343,0.230033,0.018319,0.016346,5.970663e+09
118,2010-06-23 00:00:00-04:00,8.346431,8.348863,8.143378,8.236697,768457200,0.0,0.0,AAPL,-0.010517,0.013919,0.114186,0.235667,0.018486,0.015812,6.329549e+09


In [5]:
import os
from getpass import getpass

# 1. Ask for your token securely (paste it when prompted)
token = getpass('Enter your GitHub Personal Access Token (ghp_...): ')
repo_url = f"https://{token}@github.com/brianrp09232000/multimodal-eq-sizing.git"

# 2. Clone the team repository using the token
!git clone {repo_url}

# 3. Define the correct source paths (from your Google Drive)
#    These paths match the logic from your earlier cells.
source_notebook = "/content/drive/MyDrive/leg_one_indicators/leg_one_indicators.ipynb"
source_csv = "/content/drive/MyDrive/leg_one_indicators/return_data.csv"

# 4. Move the files into the new repository folder
#    We use quotes around paths just in case there are spaces.
!cp "{source_notebook}" multimodal-eq-sizing/
!cp "{source_csv}" multimodal-eq-sizing/

# 5. Enter the repository folder
%cd multimodal-eq-sizing

# 6. Configure your Git identity
!git config user.email "fabiopach7@gmail.com"
!git config user.name "fabiopach7"

# 7. Create your branch
!git checkout -b feature/ticket-16-leg-1

# 8. Add, Commit, and Push
!git add leg_one_indicators.ipynb return_data.csv
!git commit -m "Add leg 1 indicators calculations (Closes #16)"
!git push origin feature/ticket-16-leg-1

Cloning into 'multimodal-eq-sizing'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 267 (delta 18), reused 3 (delta 3), pack-reused 227 (from 1)
Receiving objects: 100% (267/267), 80.80 KiB | 4.25 MiB/s, done.
Resolving deltas: 100% (137/137), done.
cp: cannot stat 'leg_one_indicators.ipynb': No such file or directory
cp: cannot stat 'return_data.csv': No such file or directory
/content/multimodal-eq-sizing
Switched to a new branch 'feature/ticket-16-leg-1'
fatal: pathspec 'leg_one_indicators.ipynb' did not match any files
On branch feature/ticket-16-leg-1
nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address
